<a href="https://colab.research.google.com/github/shadynagy111-eng/Decoding-EGX-Price-Action/blob/Dataset/Final_Dataset_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
AI4Fin Dataset Builder
======================
Creates a rolling-window dataset from EGX stock CSVs.

Pipeline:
  - 40-day lookback window (OHLC) → input
  - 5-day forward return           → z-score normalized → label
  - Labelling strategy: HYBRID z-score normalization
      1. Compute each company's 5-day return mean & std from TRAINING data only
      2. Z-score every window's forward return using those training stats
      3. Apply fixed z-score thresholds across ALL companies
    → Same visual pattern = same label regardless of which stock it came from
    → Corrects for secular bull/bear trends without future leakage
  - Gray zone (|z| 0.5–1.0) excluded for clean signal boundaries
  - Time-based train/val/test split — never shuffled, no leakage

Z-score thresholds (on normalized return):
  Bullish  : z ≥  +Z_BULL   (unusually strong move up)
  Bearish  : z ≤  −Z_BULL   (unusually strong move down)
  Neutral  : |z| ≤  Z_NEUTRAL (unremarkable move)
  Gray zone: Z_NEUTRAL < |z| < Z_BULL  → excluded

Output:
  dataset_metadata.csv — one row per window, includes raw pct_change,
  z_score, and all info needed for downstream image generation.
"""

import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path

# ─── CONFIG ────────────────────────────────────────────────────────────────────
DATA_DIR   = "./egx_stock_data/egx_stock_data"
OUTPUT_DIR = "./dataset"

WINDOW_SIZE  = 40   # trading days in lookback window (~2 months)
FORWARD_DAYS = 5    # trading days for forward return (~1 week)

# Z-score thresholds applied uniformly across all companies
Z_BULL    = 1.0    # |z| ≥ 1.0 → Bullish / Bearish  (~top/bottom 16% of a normal dist)
Z_NEUTRAL = 0.5    # |z| ≤ 0.5 → Neutral             (~middle 38%)
# Gray zone: 0.5 < |z| < 1.0  → excluded (~remaining 46% → keeps labels clean)

# Time-based splits (window ENTRY date determines the split)
TRAIN_END  = "2022-12-31"
VAL_START  = "2023-01-01"
VAL_END    = "2023-12-31"
TEST_START = "2024-01-01"

IMAGE_SIZE = 224   # stored in metadata; used by image-gen scripts
# ───────────────────────────────────────────────────────────────────────────────


def assign_split(date: pd.Timestamp) -> str:
    if date <= pd.Timestamp(TRAIN_END):
        return "train"
    elif pd.Timestamp(VAL_START) <= date <= pd.Timestamp(VAL_END):
        return "val"
    elif date >= pd.Timestamp(TEST_START):
        return "test"
    return "unknown"


def label_from_zscore(z: float) -> str | None:
    """Apply fixed z-score thresholds. Returns None for gray zone."""
    if z >= Z_BULL:
        return "Bullish"
    elif z <= -Z_BULL:
        return "Bearish"
    elif abs(z) <= Z_NEUTRAL:
        return "Neutral"
    return None   # gray zone → excluded


def process_ticker(filepath: str) -> pd.DataFrame:
    ticker = Path(filepath).stem.replace("_price", "")

    df = pd.read_csv(filepath, parse_dates=["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    n  = len(df)

    # ── Step 1: compute ALL 5-day forward returns (raw %) ─────────────────
    # ret[i] = return if you enter at close of row i and exit at close of row i+5
    raw_rets = np.full(n, np.nan)
    for i in range(n - FORWARD_DAYS):
        entry_c   = df.loc[i, "Close"]
        exit_c    = df.loc[i + FORWARD_DAYS, "Close"]
        raw_rets[i] = (exit_c - entry_c) / entry_c

    # ── Step 2: fit normalization stats on TRAINING rows only ─────────────
    # "Training row" = any row whose date is within the training period AND
    # whose forward window doesn't bleed past TRAIN_END (strict no-leakage).
    train_mask = (
        (df["Date"] <= pd.Timestamp(TRAIN_END)) &
        (df.index + FORWARD_DAYS < n) &
        (~np.isnan(raw_rets))
    )
    train_rets = raw_rets[train_mask]

    if len(train_rets) < 30:
        print(f"  WARNING: {ticker} has only {len(train_rets)} training returns — skipping")
        return pd.DataFrame()

    train_mean = float(np.mean(train_rets))
    train_std  = float(np.std(train_rets, ddof=1))

    if train_std == 0:
        print(f"  WARNING: {ticker} has zero std in training — skipping")
        return pd.DataFrame()

    # ── Step 3: build windows ─────────────────────────────────────────────
    records = []

    for i in range(n - WINDOW_SIZE - FORWARD_DAYS + 1):
        win_end_idx = i + WINDOW_SIZE - 1
        fwd_end_idx = i + WINDOW_SIZE + FORWARD_DAYS - 1

        if fwd_end_idx >= n:
            break

        # The forward return is measured from the LAST day of the window
        # (i.e., the model "sees" the window and bets on what happens next)
        entry_close = df.loc[win_end_idx, "Close"]
        exit_close  = df.loc[fwd_end_idx, "Close"]
        pct_change  = (exit_close - entry_close) / entry_close

        # Z-score using TRAINING stats only — same formula for all splits
        z_score = (pct_change - train_mean) / train_std

        label = label_from_zscore(z_score)
        if label is None:
            continue   # gray zone → skip

        entry_date   = df.loc[i, "Date"]
        win_end_date = df.loc[win_end_idx, "Date"]
        fwd_end_date = df.loc[fwd_end_idx, "Date"]
        fwd_start_date = df.loc[win_end_idx + 1, "Date"]

        split = assign_split(entry_date)
        if split == "unknown":
            continue

        records.append({
            "ticker":           ticker,
            "window_start":     entry_date.date(),
            "window_end":       win_end_date.date(),
            "fwd_start":        fwd_start_date.date(),
            "fwd_end":          fwd_end_date.date(),
            "entry_close":      round(entry_close, 4),
            "exit_close":       round(exit_close, 4),
            "pct_change":       round(pct_change, 6),
            "z_score":          round(z_score, 4),
            "label":            label,
            "split":            split,
            "window_row_start": i,           # row index in original CSV (for image gen)
            "window_row_end":   win_end_idx,
            "image_size":       IMAGE_SIZE,
            # Normalization stats stored for full reproducibility
            "train_mean":       round(train_mean, 8),
            "train_std":        round(train_std, 8),
        })

    return pd.DataFrame(records)


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
    if not csv_files:
        raise FileNotFoundError(f"No CSVs found in {DATA_DIR}")

    print(f"Processing {len(csv_files)} tickers...\n")

    all_frames = []
    for f in csv_files:
        ticker = Path(f).stem.replace("_price", "")
        df_out = process_ticker(f)
        all_frames.append(df_out)

        by_split = df_out.groupby(["split", "label"]).size().unstack(fill_value=0)
        print(f"  {ticker:6s}  total={len(df_out):5d}")
        for split in ["train", "val", "test"]:
            if split in by_split.index:
                row = by_split.loc[split]
                print(f"    {split:5s}: Bullish={row.get('Bullish',0):4d}  "
                      f"Bearish={row.get('Bearish',0):4d}  Neutral={row.get('Neutral',0):4d}")

    meta = pd.concat(all_frames, ignore_index=True)

    # ── Summary ────────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("DATASET SUMMARY")
    print("="*60)
    summary = meta.groupby(["split", "label"]).size().unstack(fill_value=0)
    print(summary)
    print(f"\nTotal windows (excl. gray): {len(meta):,}")

    # ── Class-balance check ────────────────────────────────────────────────
    print("\nClass balance per split:")
    for split in ["train", "val", "test"]:
        sub = meta[meta["split"] == split]
        if len(sub) == 0:
            continue
        counts = sub["label"].value_counts()
        total  = len(sub)
        print(f"  {split}: " + "  ".join(
            f"{l}={counts.get(l,0)} ({100*counts.get(l,0)/total:.1f}%)"
            for l in ["Bullish", "Bearish", "Neutral"]
        ))

    # ── Save ───────────────────────────────────────────────────────────────
    out_path = os.path.join(OUTPUT_DIR, "dataset_metadata.csv")
    meta.to_csv(out_path, index=False)
    print(f"\nSaved: {out_path}")

    # Also save per-split CSVs for convenience
    for split in ["train", "val", "test"]:
        sub = meta[meta["split"] == split]
        sub.to_csv(os.path.join(OUTPUT_DIR, f"{split}_metadata.csv"), index=False)
        print(f"Saved: {OUTPUT_DIR}/{split}_metadata.csv  ({len(sub):,} rows)")


if __name__ == "__main__":
    main()

In [ ]:
!cp "/content/drive/MyDrive/23-GP26-Seif-AI4FinEGX/Datasets/egx_stock_data.zip" "/content/"

# 2. Unzip it locally (Takes about 1-2 minutes for 56,000 files)
!unzip -q /content/egx_stock_data.zip -d /content/

In [ ]:
!pip install -q mplfinance pyts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 105.7 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplfinance as mpf
from pyts.image import GramianAngularField
from PIL import Image
from tqdm import tqdm
import warnings
from collections import defaultdict

# --- Configuration ---
METADATA_PATH = '/content/dataset_metadata.csv'
DATA_DIR = '/content/egx_stock_data/egx_stock_data'  # Based on kernel state
OUTPUT_DIR_CANDLE = '/content/candlesticks'
OUTPUT_DIR_HEATMAP = '/content/heatmaps'
IMAGE_SIZE = (224, 224)
WINDOW_SIZE = 40

# --- 1. Create Directory Trees ---
def create_dir_tree(base_dir):
    splits = ['train', 'val', 'test']
    labels = ['Bullish', 'Bearish', 'Neutral']
    for split in splits:
        for label in labels:
            os.makedirs(os.path.join(base_dir, split, label), exist_ok=True)

create_dir_tree(OUTPUT_DIR_CANDLE)
create_dir_tree(OUTPUT_DIR_HEATMAP)

# --- 2. Load Metadata ---
try:
    metadata_df = pd.read_csv(METADATA_PATH)
except FileNotFoundError:
    raise FileNotFoundError(f"Metadata file not found at {METADATA_PATH}")

# --- 3. Cache DataFrames ---
stock_cache = {}

def get_stock_data(ticker):
    if ticker not in stock_cache:
        file_path = os.path.join(DATA_DIR, f"{ticker}_price.csv")
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            if 'Date' in df.columns:
                df['Date'] = pd.to_datetime(df['Date'])
                df.set_index('Date', inplace=True)
            elif 'date' in df.columns:
                df['date'] = pd.to_datetime(df['date'])
                df.set_index('date', inplace=True)
            # Ensure required columns are present and numeric
            for col in ['Open', 'High', 'Low', 'Close']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
            stock_cache[ticker] = df
        else:
            stock_cache[ticker] = None # Mark as not found to avoid repeated checks
    return stock_cache[ticker]

# --- 4. Image Generation Functions ---

# 4a. Candlestick Generation
def generate_candlestick(window_df, save_path):
    # Need to temporarily disable interactive plotting to avoid displaying 1000s of plots
    plt.ioff()

    # mplfinance requires specific column names (capitalized)
    plot_df = window_df[['Open', 'High', 'Low', 'Close']].copy()

    # Convert index to DatetimeIndex if it isn't already
    if not isinstance(plot_df.index, pd.DatetimeIndex):
        # fallback, though get_stock_data should handle this
        plot_df.index = pd.to_datetime(plot_df.index)

    # Create figure and axes using returnfig=True
    fig, axes = mpf.plot(
        plot_df,
        type='candle',
        style='charles',
        volume=False,
        axisoff=True,
        returnfig=True,
        figsize=(2.24, 2.24), # 224x224 at 100 dpi
    )

    # Strip all chrome and save
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.savefig(save_path, dpi=100, bbox_inches='tight', pad_inches=0, transparent=False)
    plt.close(fig)

# 4b. Heatmap Generation
gasf_transformer = GramianAngularField(image_size=WINDOW_SIZE, method='summation')

def normalize_series(series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return np.zeros_like(series)
    return 2 * (series - min_val) / (max_val - min_val) - 1

def generate_heatmap(window_df, save_path):
    # Extract CULR components
    O = window_df['Open'].values
    H = window_df['High'].values
    L = window_df['Low'].values
    C = window_df['Close'].values

    # C = Close price
    # U = Upper shadow = High - max(Open, Close)
    # L_shadow = Lower shadow = min(Open, Close) - Low
    # R = Range = High - Low

    C_series = C
    U_series = H - np.maximum(O, C)
    L_series = np.minimum(O, C) - L
    R_series = H - L

    # Normalize
    C_norm = normalize_series(C_series)
    U_norm = normalize_series(U_series)
    L_norm = normalize_series(L_series)
    R_norm = normalize_series(R_series)

    # Apply GASF (requires 2D input (n_samples, n_features))
    C_gasf = gasf_transformer.transform(C_norm.reshape(1, -1))[0]
    U_gasf = gasf_transformer.transform(U_norm.reshape(1, -1))[0]
    L_gasf = gasf_transformer.transform(L_norm.reshape(1, -1))[0]
    R_gasf = gasf_transformer.transform(R_norm.reshape(1, -1))[0]

    # Combine into RGB channels
    # R channel = C (Close)
    # G channel = 0.5 * U + 0.5 * L
    # B channel = R (Range)

    R_channel = C_gasf
    G_channel = 0.5 * U_gasf + 0.5 * L_gasf
    B_channel = R_gasf

    # Stack to (40, 40, 3)
    rgb_array = np.stack([R_channel, G_channel, B_channel], axis=-1)

    # Normalize to [0, 255] uint8
    min_val = rgb_array.min()
    max_val = rgb_array.max()
    if max_val > min_val:
        img_norm = (rgb_array - min_val) / (max_val - min_val)
    else:
        img_norm = np.zeros_like(rgb_array)

    img_uint8 = (img_norm * 255).astype(np.uint8)

    # Resize to 224x224 using BICUBIC
    img_pil = Image.fromarray(img_uint8).resize(IMAGE_SIZE, Image.BICUBIC)

    # Save
    img_pil.save(save_path)

# --- 5. Main Loop ---
print("Starting image generation...")

# Track statistics
stats = {
    'candlestick': defaultdict(lambda: defaultdict(int)),
    'heatmap': defaultdict(lambda: defaultdict(int))
}
missing_csvs = set()

for index, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Processing Windows"):
    ticker = row['ticker']
    start_idx = int(row['window_row_start'])
    end_idx = int(row['window_row_end'])
    label = row['label']
    split = row['split']

    # Ensure valid split and label to avoid directory errors
    if split not in ['train', 'val', 'test'] or label not in ['Bullish', 'Bearish', 'Neutral']:
        warnings.warn(f"Invalid split '{split}' or label '{label}' at row {index}. Skipping.")
        continue

    df = get_stock_data(ticker)
    if df is None:
        if ticker not in missing_csvs:
            warnings.warn(f"Stock data for {ticker} not found. Skipping related rows.")
            missing_csvs.add(ticker)
        continue

    # Slice window (inclusive)
    # Note: Using .iloc for row-based indexing as specified
    try:
        window_df = df.iloc[start_idx:end_idx+1]
    except IndexError:
         warnings.warn(f"Index error slicing {ticker} from {start_idx} to {end_idx}. Skipping.")
         continue

    if len(window_df) != WINDOW_SIZE:
         warnings.warn(f"Window size for {ticker} is {len(window_df)}, expected {WINDOW_SIZE}. Skipping row {index}.")
         continue

    # Define filenames and paths
    filename = f"{ticker}_{start_idx}_{end_idx}.png"
    candle_path = os.path.join(OUTPUT_DIR_CANDLE, split, label, filename)
    heatmap_path = os.path.join(OUTPUT_DIR_HEATMAP, split, label, filename)

    # Generate and save Candlestick
    try:
        generate_candlestick(window_df, candle_path)
        stats['candlestick'][split][label] += 1
    except Exception as e:
        warnings.warn(f"Error generating candlestick for {ticker} window {start_idx}-{end_idx}: {e}")

    # Generate and save Heatmap
    try:
        generate_heatmap(window_df, heatmap_path)
        stats['heatmap'][split][label] += 1
    except Exception as e:
        warnings.warn(f"Error generating heatmap for {ticker} window {start_idx}-{end_idx}: {e}")

# --- 6. Summary ---
print("\n=== Generation Summary ===")
for img_type in ['candlestick', 'heatmap']:
    print(f"\n{img_type.capitalize()} Images:")
    total_type = 0
    for split in ['train', 'val', 'test']:
        print(f"  {split.capitalize()}:")
        split_total = 0
        for label in ['Bullish', 'Bearish', 'Neutral']:
            count = stats[img_type][split][label]
            print(f"    {label}: {count}")
            split_total += count
        print(f"  Total {split}: {split_total}")
        total_type += split_total
    print(f"Total {img_type} images generated: {total_type}")

if missing_csvs:
    print(f"\nWarning: Missing CSV files for tickers: {', '.join(missing_csvs)}")
else:
    print("\nAll stock CSVs found successfully.")


Starting image generation...


Processing Windows: 100%|██████████| 48033/48033 [39:07<00:00, 20.46it/s]


=== Generation Summary ===

Candlestick Images:
  Train:
    Bullish: 5617
    Bearish: 4932
    Neutral: 23758
  Total train: 34307
  Val:
    Bullish: 1244
    Bearish: 724
    Neutral: 2859
  Total val: 4827
  Test:
    Bullish: 1632
    Bearish: 1271
    Neutral: 5996
  Total test: 8899
Total candlestick images generated: 48033

Heatmap Images:
  Train:
    Bullish: 5617
    Bearish: 4932
    Neutral: 23758
  Total train: 34307
  Val:
    Bullish: 1244
    Bearish: 724
    Neutral: 2859
  Total val: 4827
  Test:
    Bullish: 1632
    Bearish: 1271
    Neutral: 5996
  Total test: 8899
Total heatmap images generated: 48033

All stock CSVs found successfully.


In [ ]:
from google.colab import drive

# Mount Google Drive (if not already mounted)
drive.mount('/content/drive')

# Zip the datasets quietly
print("Zipping candlesticks...")
!zip -r -q /content/candlesticks.zip /content/candlesticks

print("Zipping heatmaps...")
!zip -r -q /content/heatmaps.zip /content/heatmaps

# Copy to Google Drive
print("Copying to Google Drive...")
!cp /content/candlesticks.zip /content/drive/MyDrive/
!cp /content/heatmaps.zip /content/drive/MyDrive/

print("Done! Check your Google Drive for the zip files.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Zipping candlesticks...
Zipping heatmaps...
Copying to Google Drive...
Done! Check your Google Drive for the zip files.
